# Assignment 3: Build OpenEvolve

[AlphaEvolve](https://arxiv.org/abs/2506.13131) improves a program by running an LLM in an evolutionary loop, and [OpenEvolve](https://github.com/algorithmicsuperintelligence/openevolve) is its open-source implementation. You implement the core of each component, close the loop, and run it on the paper's circle-packing task: 26 circles in the unit square, maximize the sum of the radii (AlphaEvolve reached 2.635).

Every part has the same shape. Write the formula, then fill in the function by following its numbered comments. **Try it** runs your code and OpenEvolve's on a tiny example and prints both, which is handy while you are debugging. The **check** cell then compares them on many cases. Code that is not part of the algorithm (config, LLM calls, plotting, the checks) is in `oe_support.py`.

In [ ]:
%pip install -q openevolve openai
import os, re, random, uuid
# Colab: upload oe_support.py, circle_packing_evaluator.py, initial_program.py if they are not here yet
if not all(os.path.exists(f) for f in ("oe_support.py", "circle_packing_evaluator.py", "initial_program.py")):
    from google.colab import files; files.upload()


os.environ.setdefault("OPENAI_API_KEY", "")   # paste your key here for Part 9 (not needed before)
cfg = oe.make_config(iterations=40)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.5/129.5 kB 2.0 MB/s eta 0:00:00


## The loop (AlphaEvolve, Figure 2)

Parts 1 to 7 build the pieces. Part 8 connects them in this order.

![alphaevolve_fig2.png](attachment:alphaevolve_fig2.png)

## 1. Apply the LLM's edit: `apply_diff` (§2.3)

> *the code between `<<<<<<< SEARCH` and `=======` is the exact segment to match in the current program version. The code between `=======` and `>>>>>>> REPLACE` is the new segment that will replace the original one.*

**Formula:** *(your answer)*

In [ ]:
DIFF = r"<<<<<<< SEARCH\n(.*?)=======\n(.*?)>>>>>>> REPLACE"

def apply_diff(code: str, diff_text: str) -> str:
    """Apply every SEARCH/REPLACE block in diff_text to code, in order. Returns the new code."""
    # 1. Work on a list of lines.
    # 2. Find all blocks: re.findall(DIFF, diff_text, re.DOTALL) gives (search, replace) pairs.
        # 3. rstrip() each side (the regex keeps a trailing newline), then split into lines.
        # 4. Find the FIRST position j where the next len(s) lines equal s exactly;
        #    replace that slice with r and stop looking. No match anywhere -> skip this block silently.
    # 5. Join the lines back with "\n".
    raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_apply_diff(apply_diff)

In [ ]:
oe.check_apply_diff(apply_diff)

## 2. Evaluate in stages: `cascade_evaluate` (§2.1, §2.4)

> *$h$ maps a solution to a set of scalar evaluation metrics … new solutions are evaluated on the next stage only if they achieve sufficiently promising results in all earlier stages.*

**Formula:** *(your answer)*

In [ ]:
def cascade_evaluate(path, stages, thresholds):
    """stages = [stage1, stage2, ...], each stage(path) -> dict or EvaluationResult(metrics, artifacts).
    thresholds[s-1] must be passed before stage s+1 may run. Returns an EvaluationResult."""
    # 1. Run stage 1 (wrap its output with oe.as_result).
    #    If it raises: return oe.EvaluationResult(metrics={"stage1_passed": 0.0, "error": 0.0}, artifacts={"stderr": str(e)}).
        # 2. Gate: stop and return res if there is no threshold for this stage,
        #    or if not oe.passes_threshold(res.metrics, thresholds[s-1])   (it looks at combined_score).
        # 3. Run the next stage. If it raises: set res.metrics[f"stage{s+1}_passed"] = 0.0,
        #    res.artifacts[f"stage{s+1}_stderr"] = str(e), and return res (earlier results are kept).
        # 4. Merge: numeric metrics only, drop the key "error", cast to float, later stage overwrites.
        #    Artifacts: {**old, **new}.
    raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_cascade(cascade_evaluate)

In [ ]:
oe.check_cascade(cascade_evaluate)
print(oe.evaluate_program(oe.INITIAL_PROGRAM, cascade_evaluate, cfg))   # the real task: initial program

## 3. Store programs with MAP-Elites: `add` (§2.5)

> *the evolutionary database implements an algorithm that is inspired by a combination of the MAP elites algorithm and island-based population models.*

**Formula:** *(your answer)*

![o3_map_elites.png](attachment:o3_map_elites.png)

In [ ]:
class ProgramDatabase(oe.BaseDatabase):
    # Given by OpenEvolve (self.…): programs {id: Program}, islands [set of ids], island_feature_maps [{cell key: id}],
    # archive (set of ids), _calculate_feature_coords, _feature_coords_to_key, _is_better, _update_archive,
    # _enforce_population_limit, _update_best_program, _update_island_best_program.

    def add(self, program, iteration=None, target_island=None):
        # 1. Store the program. If iteration is given: program.iteration_found = iteration and
        #    self.last_iteration = max(self.last_iteration, iteration).
        # 2. Its MAP-Elites cell: coords = self._calculate_feature_coords(program)  (after storing it).
        # 3. Island k: target_island if given; else the parent's island (parent.metadata["island"]) if the parent
        #    exists and has one; else self.current_island. Then k %= number of islands.
        # 4. MAP-Elites rule on island k: key = self._feature_coords_to_key(coords), old = current elite of that cell.
        #    The new program takes the cell if the cell is empty, the old elite no longer exists, or
        #    self._is_better(program, old elite). When it replaces an old elite:
        #      - if the old elite is in self.archive, swap it out of the archive for the new program;
        #      - remove the old elite from self.islands[k].
        # 5. Always (even if it lost its cell): add to self.islands[k], set program.metadata["island"] = k, then call
        #    _update_archive(program), _enforce_population_limit(exclude_program_id=program.id),
        #    _update_best_program(program), _update_island_best_program(program, k). Return program.id.
        raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_add(ProgramDatabase)

In [ ]:
oe.check_add(ProgramDatabase)

## 4. Pick a parent and inspirations: `sample_from_island` (§2.5)

> *A key challenge … is balancing exploration and exploitation, to continuously improve the best programs while maintaining diversity.*

**Formula:** *(your answer)*

In [ ]:
class ProgramDatabase(ProgramDatabase):
    def sample_from_island(self, island_id, num_inspirations=3):
        """Return (parent, inspirations), all taken from island `island_id`."""
        # 1. ids = the island's program ids that still exist in self.programs.
        #    Empty island -> return self.sample(num_inspirations)   (OpenEvolve's global fallback).
        # 2. u = random.random(); elites = archive ids that still exist.
        # 3a. u < exploration_ratio (0.2): explore -> uniform over ids (random.choice).
        # 3b. u < exploration + exploitation (0.9) and elites exist: exploit -> uniform over the elites on this
        #     island (metadata["island"] == island_id); if there are none, uniform over all elites.
        # 3c. Otherwise: fitness-proportional over the island, weight = max(oe.fitness(metrics), 0.001),
        #     one draw with random.choices(programs, weights=w)[0].
        # 4. Inspirations: the other ids on the island; take all if fewer than num_inspirations,
        #    else random.sample(others, num_inspirations). Return (parent, [Program, ...]).
        raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_sampling(ProgramDatabase)

In [ ]:
oe.check_sampling(ProgramDatabase)

## 5. Build the prompt: `build_prompt` (§2.2)

> *This prompt comprises multiple previously discovered solutions sampled from the program database, as well as system instructions … [and] rendered evaluation results.*

**Formula:** *(your answer)*

In [ ]:
class PromptSampler(oe.BasePromptSampler):
    # Given by OpenEvolve: self.template_manager.get_template(name), self._apply_template_variations(t),
    # self._format_metrics(m), self._identify_improvement_areas(code, code, m, previous, dims),
    # self._format_evolution_history(previous, top, inspirations, language, dims), self._render_artifacts(a).

    def build_prompt(self, current_program, program_metrics, previous_programs, top_programs, inspirations,
                     language="python", program_artifacts=None, feature_dimensions=()):
        # 1. Template: self.template_manager.get_template("diff_user"), passed through
        #    self._apply_template_variations (stochastic formatting; a no-op with the default config).
        # 2. Fill every field of the template with .format(...):
        #      metrics            <- self._format_metrics(program_metrics)
        #      fitness_score      <- f"{oe.fitness(program_metrics, dims):.4f}"
        #      feature_coords     <- oe.format_feature_coordinates(program_metrics, dims)
        #      feature_dimensions <- ", ".join(dims)
        #      improvement_areas  <- "fitness improved/declined vs. previous": self._identify_improvement_areas(...)
        #      evolution_history  <- previous attempts + top programs + inspirations: self._format_evolution_history(...)
        #      current_program, language
        #      artifacts          <- self._render_artifacts(program_artifacts) if there are any, else ""
        # 3. Return {"system": self.config.system_message, "user": user}.
        raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_prompt(PromptSampler)

In [ ]:
oe.check_prompt(PromptSampler, ProgramDatabase)

## 6. Choose an LLM: `sample_model` (§2.3)

> *a combination of Gemini 2.0 Flash and Gemini 2.0 Pro … Flash enables a higher rate of candidate generation … Pro provides occasional, higher-quality suggestions.*

**Formula:** *(your answer)*

In [ ]:
def sample_model(weights, rng):
    """weights: normalized model weights, e.g. [0.8, 0.2]; rng: a random.Random. Return the index of the model to call."""
    # 1. One weighted draw of an index: rng.choices(range(len(weights)), weights=weights, k=1)[0].
    #    (OpenEvolve uses its own random.Random, so model choice doesn't disturb the database's random stream.)
    raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_sample_model(sample_model)

In [ ]:
oe.check_sample_model(sample_model)

## 7. Migrate between islands: `migrate_programs` (§2.5)

> *…inspired by a combination of the MAP elites algorithm and island-based population models.* Islands evolve separately; every `migration_interval` generations each island sends copies of its best programs to its two neighbours on a ring.

**Formula:** *(your answer)*

![o7_migration.png](attachment:o7_migration.png)

In [ ]:
class ProgramDatabase(ProgramDatabase):
    def migrate_programs(self):
            # 1. The island's programs sorted by fitness, best first (oe.fitness(p.metrics, self.config.feature_dimensions)).
            # 2. Migrants = the top max(1, int(len(progs) * self.config.migration_rate)); skip empty islands.
                # 3. A program that is itself a migrant copy (metadata["migrant"]) never migrates again.
                # 4. Targets: (i + 1) % K, then (i - 1) % K. Skip a target that already has a program with identical code.
                    # 5. Copy = oe.Program(id=str(uuid.uuid4()), code, language, parent_id=p.id, generation,
                    #    metrics=p.metrics.copy(), metadata={**p.metadata, "island": j, "migrant": True});
                    #    insert with self.add(copy, target_island=j)   (it must win its MAP-Elites cell like any child).
        # 6. Remember when we migrated: self.last_migration_generation = max(self.island_generations).
        raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it on a tiny example: your output next to OpenEvolve's.
oe.demo_migration(ProgramDatabase)

In [ ]:
oe.check_migration(ProgramDatabase)

## 8. Close the loop: `OpenEvolveLoop`

> *The Prompt sampler uses programs from the Program database to construct rich prompts. Given these prompts, the LLMs generate code modifications (diffs), which are applied to create new programs. These are then scored by Evaluators, and promising solutions are registered back into the Program database.* (Figure 2 caption)

**Formula:** *(your answer)*

In [ ]:
class OpenEvolveLoop:
    def __init__(self, cfg, llm):
        self.cfg, self.llm, self.history = cfg, llm, []
        self.db = ProgramDatabase(cfg.database)
        self.sampler = PromptSampler(cfg.prompt)

    def evaluate(self, code):                       # -> (metrics, artifacts), using your cascade
        return oe.evaluate_program(code, cascade_evaluate, self.cfg)

    def run_iteration(self, t, k):
        """One pass of Figure 2 on island k. Returns (child Program, artifacts) or (None, None)."""
        # 1. Sample: parent, inspirations = self.db.sample_from_island(k, n), with n = cfg.prompt.num_top_programs (3).
        # 2. Context: the parent's island, ranked by metrics["combined_score"] (best first).
        #    previous = the top n, top = the top n + cfg.prompt.num_diverse_programs; pass them as p.to_dict().
        # 3. Prompt: self.sampler.build_prompt(parent code, parent metrics, previous, top, inspirations, "python",
        #    the parent's artifacts self.db.get_artifacts(parent.id) or None, cfg.database.feature_dimensions).
        # 4. LLM: reply = self.llm.generate(system, user). No diff block in the reply -> return None, None.
        # 5. Child = apply_diff(parent.code, reply); metrics, artifacts = self.evaluate(child code).
        # 6. Wrap it: oe.Program(id=uuid4, code, parent_id=parent.id, generation=parent.generation + 1, metrics,
        #    iteration_found=t, metadata={"changes": oe.format_diff_summary(blocks), "parent_metrics": parent.metrics,
        #    "island": island}). Return (child, artifacts).
        raise NotImplementedError()  # YOUR CODE HERE

    def run_evolution(self, T, verbose=True):
        # 1. Evaluate the initial program oe.INITIAL_PROGRAM, wrap it in oe.Program, self.db.add it (lands on island 0),
        #    and self.db.store_artifacts(its id, artifacts).
            # 2. Round-robin over islands: k = (t - 1) % K; child, artifacts = self.run_iteration(t, k).
            # 3. If there is a child: self.db.add(child, iteration=t, target_island=k); store its artifacts;
            #    self.db.increment_island_generation(k); if self.db.should_migrate(): self.db.migrate_programs().
            # 4. Every iteration: oe.log(self, t, k, child, migrated, verbose)   (history for plots and checks).
        # 5. Return self.db.get_best_program().
        raise NotImplementedError()  # YOUR CODE HERE


In [ ]:
# Try it: 4 iterations with the offline mock LLM (it randomly perturbs one constant; free, a few seconds).
random.seed(0)
tiny = OpenEvolveLoop(oe.make_config(iterations=4), oe.make_ensemble(cfg, sample_model, mock=True))
tiny.run_evolution(4)

In [ ]:
oe.check_loop(OpenEvolveLoop)

## 9. Run it, and compare with OpenEvolve

The first cell uses the offline mock LLM. For real models, paste an API key in the setup cell and set `mock=False`: 40 iterations take about 5 minutes. The second cell runs the official `openevolve-run` on the same program, evaluator and config (about 12 minutes), then plots both curves together. In our test run, both got to a sum of radii of about 1.76 in 40 iterations (AlphaEvolve's result is 2.635).

In [ ]:
random.seed(0)
loop = OpenEvolveLoop(cfg, oe.make_ensemble(cfg, sample_model, mock=True))   # mock=False: real LLMs
best = loop.run_evolution(cfg.max_iterations)
oe.plot_run(loop)

In [ ]:
oe.write_cli_config(cfg)
!openevolve-run initial_program.py circle_packing_evaluator.py --config openevolve_config.yaml --output openevolve_output
oe.plot_run(loop, other=oe.cli_curve("openevolve_output"))

## Questions

**Q1.** Your loop and `openevolve-run` use the same algorithm and config. Why do their curves still differ?

*(your answer)*

**Q2.** Rerun Part 9 twice: once with `exploration_ratio = 0` and `exploitation_ratio = 1` (in `cfg.database`), once with `num_islands = 1`. Compare the best score and the number of occupied cells (`sum(map(len, loop.db.island_feature_maps))`), and explain the difference using the formula from Part 4.

*(your answer)*

**Q3.** In the course framework (what is evolved · what signal drives it · how the change is made), where does OpenEvolve sit? What would have to change for it to evolve the harness?

*(your answer)*